# Домашнее задание 1: Промпт-инжиниринг

В этом задании вы потренируетесь:
1. Экспериментировать с параметрами генерации (`temperature`, `top_p`)
2. Применять техники промпт-инжиниринга (CoT, Few-shot, уточнение правил) к задаче извлечения с подвохами
3. Разбираться, что дают reasoning-модели и их параметры (`reasoning_effort`, `max_tokens`, рассуждения модели)
4. Получать структурированный ответ с помощью Pydantic
5. Применять паттерны Schema-Guided Reasoning (каскад, маршрутизация, цикл)
6. Проектировать собственную SGR-схему с нуля

| Задача | Баллы |
|---|---|
| 1. Параметры генерации | 0.5 |
| 2А. Промпт-инжиниринг: извлечение из тикетов | 1.5 |
| 2Б. Reasoning-модели и их параметры | 1 |
| 3. Structured output: вакансия | 1.5 |
| 4. SGR: планировщик путешествий | 2 |
| 5. Своя SGR-схема: поездка компании друзей | 2.5 |
| **Итого** | **10** |

Все запросы идут через библиотеку `openai` и функции из семинара 2. Основной провайдер — Neural Deep; под вызовами закомментированы альтернативы.

In [1]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()

os.environ["OPENROUTER_API_KEY"] = user_secrets.get_secret("OPENROUTER_API_KEY")

print("OpenRouter key loaded")

OpenRouter key loaded


## 0. Подготовка

Код ниже — тот же, что в разделе 0 семинара 2: клиенты-синглтоны на провайдера и функции `chat` / `call_llm`.

In [2]:
!pip install -q openai==3.19.0 python-dotenv==1.2.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 28.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.0/345.0 kB 11.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.85.7 requires openai<3.0.0,>=2.20.0, but you have openai 3.19.0 which is incompatible.


In [12]:
import os
import json
import re
import time
from functools import lru_cache, partial
from typing import List, Literal, Optional, Union

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

In [ ]:
# load_dotenv()  # локально подтянет ключи из .env; в Colab ничего не сделает


# def get_secret(name: str) -> str | None:
#     # В Colab — из секретов (🔑 на левой панели), локально — из переменных окружения
#     try:
#         from google.colab import userdata
#     except ImportError:
#         return os.environ.get(name)
#     try:
#         return userdata.get(name)
#     except Exception:  # секрет не создан или нет доступа (Notebook access)
#         return None


# for key in ["NEURAL_DEEP_API_KEY", "LLM7_API_KEY", "MISTRAL_API_KEY",
#             "YANDEX_API_KEY", "YANDEX_FOLDER_ID", "DASHSCOPE_API_KEY"]:
#     print(f"{'✅' if get_secret(key) else '❌'} {key}")

In [7]:
PROVIDERS = {
    "neuraldeep": {"base_url": "https://api.neuraldeep.tech/v1", "api_key": "NEURAL_DEEP_API_KEY"},
    "llm7":       {"base_url": "https://api.llm7.io/v1",         "api_key": "LLM7_API_KEY"},
    "mistral":    {"base_url": "https://api.mistral.ai/v1",      "api_key": "MISTRAL_API_KEY"},
    "yandex":     {"base_url": "https://ai.api.cloud.yandex.net/v1", "api_key": "YANDEX_API_KEY"},
    "alibaba":    {"base_url": "https://ws-lqopykemg5sjxdjs.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1",
                   "api_key": "DASHSCOPE_API_KEY"},
    "openrouter": {"base_url": "https://openrouter.ai/api/v1","api_key": "OPENROUTER_API_KEY"},
}


@lru_cache(maxsize=None)
# def get_client(provider: str) -> OpenAI:
#     # Клиент создаётся при первом обращении к провайдеру, дальше возвращается тот же объект
#     config = PROVIDERS[provider]
#     api_key = get_secret(config["api_key"])
#     if not api_key:
#         raise ValueError(f"❌ {config['api_key']} не найден (секреты Colab или .env)")
#     # timeout: бесплатные тарифы иногда «подвешивают» запрос — лучше упасть и повторить, чем ждать 10 минут
#     return OpenAI(base_url=config["base_url"], api_key=api_key, timeout=300)
def get_secret(name: str) -> str | None:
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

In [8]:
client = get_client("openrouter")

In [9]:
def build_messages(
    prompt: str,
    system_prompt: str | None = None
) -> list[dict]:

    messages = []

    if system_prompt:
        messages.append({
            "role": "system",
            "content": system_prompt
        })

    messages.append({
        "role": "user",
        "content": prompt
    })

    return messages


def is_pydantic_model(obj) -> bool:
    return (
        isinstance(obj, type)
        and issubclass(obj, BaseModel)
    )


def chat(
    provider: str,
    model: str,
    prompt: str | None = None,
    system_prompt: str | None = None,
    messages: list[dict] | None = None,
    response_format=None,
    schema_in_prompt: bool = True,
    **params
):
    client = get_client(provider)

    if provider == "yandex":
        model = f"gpt://{get_secret('YANDEX_FOLDER_ID')}/{model}"

    if messages is None:
        messages = build_messages(
            prompt,
            system_prompt
        )

    if is_pydantic_model(response_format):

        schema = response_format.model_json_schema()

        if schema_in_prompt:
            schema_text = json.dumps(
                schema,
                ensure_ascii=False
            )

            messages = messages + [{
                "role": "system",
                "content":
                    "Ответь JSON-объектом строго по этой JSON-схеме:\n"
                    + schema_text
            }]

        if provider == "openrouter":

            params["response_format"] = {
                "type": "json_schema",
                "json_schema": {
                    "name": response_format.__name__,
                    "strict": True,
                    "schema": schema
                }
            }

            return client.chat.completions.create(
                model=model,
                messages=messages,
                **params
            )

        return client.chat.completions.parse(
            model=model,
            messages=messages,
            response_format=response_format,
            **params
        )

    if response_format is not None:
        params["response_format"] = response_format

    return client.chat.completions.create(
        model=model,
        messages=messages,
        **params
    )


def call_llm(
    provider: str,
    model: str,
    prompt: str | None = None,
    system_prompt: str | None = None,
    **kwargs
):
    response = chat(
        provider,
        model,
        prompt,
        system_prompt,
        **kwargs
    )

    message = response.choices[0].message

    response_format = kwargs.get("response_format")

    if is_pydantic_model(response_format):

        if getattr(message, "parsed", None) is not None:
            return message.parsed

        return response_format.model_validate_json(
            message.content
        )

    return message.content


call_openrouter = partial(call_llm, "openrouter")

call_neuraldeep = partial(call_llm, "neuraldeep")
call_llm7 = partial(call_llm, "llm7")
call_mistral = partial(call_llm, "mistral")
call_yandex = partial(call_llm, "yandex")
call_alibaba = partial(call_llm, "alibaba")
# def build_messages(prompt: str, system_prompt: str | None = None) -> list[dict]:
#     messages = []
#     if system_prompt:
#         messages.append({"role": "system", "content": system_prompt})
#     messages.append({"role": "user", "content": prompt})
#     return messages


# def is_pydantic_model(obj) -> bool:
#     return isinstance(obj, type) and issubclass(obj, BaseModel)


# def chat(provider: str, model: str, prompt: str | None = None, system_prompt: str | None = None,
#          messages: list[dict] | None = None, response_format=None, schema_in_prompt: bool = True, **params):
#     # Единая точка входа: возвращает полный объект ответа API
#     client = get_client(provider)
#     if provider == "yandex":  # YandexGPT ждёт модель в формате gpt://<folder_id>/<model>
#         model = f"gpt://{get_secret('YANDEX_FOLDER_ID')}/{model}"
#     if messages is None:
#         messages = build_messages(prompt, system_prompt)

#     if is_pydantic_model(response_format):  # structured output по Pydantic-схеме
#         if schema_in_prompt:  # явно показываем схему модели (см. раздел 5 семинара)
#             schema = json.dumps(response_format.model_json_schema(), ensure_ascii=False)
#             messages = messages + [{"role": "system", "content": f"Ответь JSON-объектом строго по этой JSON-схеме:\n{schema}"}]
#         return client.chat.completions.parse(model=model, messages=messages,
#                                              response_format=response_format, **params)
#     if response_format is not None:  # например, {"type": "json_object"}
#         params["response_format"] = response_format
#     return client.chat.completions.create(model=model, messages=messages, **params)


# def call_llm(provider: str, model: str, prompt: str | None = None, system_prompt: str | None = None, **kwargs):
#     # Текст ответа или, если передан Pydantic-класс в response_format, объект этого класса
#     message = chat(provider, model, prompt, system_prompt, **kwargs).choices[0].message
#     return message.parsed if is_pydantic_model(kwargs.get("response_format")) else message.content


# call_neuraldeep = partial(call_llm, "neuraldeep")
# call_llm7 = partial(call_llm, "llm7")
# call_mistral = partial(call_llm, "mistral")
# call_yandex = partial(call_llm, "yandex")
# call_alibaba = partial(call_llm, "alibaba")

In [10]:
result = call_openrouter(
    "openai/gpt-4o-mini",
    "Ответь одним словом: сколько будет 2 + 2?"
)

print(result)

Четыре.


### Модели

Те же, что в семинаре:

| Константа | Модель | Рассуждения |
|---|---|---|
| `MODEL` | `qwen3.6-35b-a3b-noreason` | нет — основная модель |
| `THINKING_MODEL` | `qwen3.6-35b-a3b` | да — для structured output и SGR |
| `REASONING_MODEL` | `qwen3.6-35b-a3b` | да |
| `EFFORT_MODEL` | `gpt-oss-120b` | да — поддерживает параметр `reasoning_effort` |

Альтернативы у других провайдеров: `call_mistral("ministral-14b-latest", ...)`, `call_llm7("codestral-latest", ...)`, `call_alibaba("qwen3.7-flash-2026-07-15", ...)`.

In [11]:
# MODEL = "qwen3.6-35b-a3b-noreason"      # основная модель
# THINKING_MODEL = "qwen3.6-35b-a3b"      # Qwen с рассуждениями: structured output и SGR
# REASONING_MODEL = "qwen3.6-35b-a3b" # Qwen с рассуждениями
# EFFORT_MODEL = "gpt-oss-120b"       # reasoning-модель, которая понимает reasoning_effort
MODEL = "openai/gpt-4o"
THINKING_MODEL = "openai/gpt-4o"
REASONING_MODEL = "openai/gpt-4o"
EFFORT_MODEL = "openai/gpt-4o"

In [12]:
# Вспомогательные функции

def get_json_from_result(s: str, is_list: bool = False):
    # Ищем в тексте первый JSON-объект (или список) и пытаемся его разобрать
    pattern = re.compile(r"\[.*\]" if is_list else r"\{.*\}", re.DOTALL)
    json_match = pattern.search(s or "")
    if json_match is None:
        return None
    try:
        return json.loads(json_match.group(0), strict=False)
    except json.JSONDecodeError as e:
        print(f"Не удалось разобрать JSON: {e}")
        return None


def flatten(d: dict, prefix: str = "") -> dict:
    # {"a": {"b": 1}} -> {"a.b": 1}
    items = {}
    for k, v in d.items():
        key = f"{prefix}.{k}" if prefix else k
        if isinstance(v, dict):
            items.update(flatten(v, key))
        else:
            items[key] = v
    return items


def values_match(expected, got) -> bool:
    # expected может быть множеством допустимых вариантов: {"fintech", "финтех"}
    if isinstance(expected, set):
        return any(values_match(e, got) for e in expected)
    if isinstance(expected, str) and isinstance(got, str):
        return expected.lower().strip() == got.lower().strip()
    return expected == got


def compare_json(result, expected: dict, skip_fields=None) -> dict:
    """Сравнивает результат (JSON-строку, dict или Pydantic-объект) с правильным ответом.
    Поля из skip_fields не сравниваются — их нужно проверить глазами. Возвращает словарь ошибок."""
    skip_fields = skip_fields or []
    if isinstance(result, str):
        result = get_json_from_result(result)
    elif isinstance(result, BaseModel):
        result = result.model_dump()
    if result is None:
        print("❌ Не удалось достать JSON из ответа")
        return {"_": "нет JSON"}

    flat_result, flat_expected = flatten(result), flatten(expected)
    correct, wrong, skipped = [], {}, []
    for key, expected_val in flat_expected.items():
        if any(skip in key for skip in skip_fields):
            skipped.append((key, flat_result.get(key, "НЕТ ПОЛЯ")))
            continue
        got = flat_result.get(key, "НЕТ ПОЛЯ")
        if values_match(expected_val, got):
            correct.append((key, got))
        else:
            wrong[key] = (expected_val, got)

    print(f"Верно: {len(correct)}/{len(correct) + len(wrong)}")
    for key, val in correct:
        print(f"  ✅ {key}: {val!r}")
    for key, (exp, got) in wrong.items():
        print(f"  ❌ {key}: ожидалось {exp!r}, получено {got!r}")
    for key, val in skipped:
        print(f"  👀 {key}: {val!r}  ← проверьте вручную")
    return wrong


def pretty_print(obj, indent: int = 0, name: str | None = None):
    """Печатает Pydantic-объект (или dict/list) в виде дерева."""
    prefix = "  " * indent
    connector = "├── " if indent > 0 else ""
    if isinstance(obj, BaseModel):
        print(f"{prefix}{connector}{name or obj.__class__.__name__}:")
        for field_name in type(obj).model_fields:
            pretty_print(getattr(obj, field_name), indent + 1, name=field_name)
    elif isinstance(obj, list) and obj and isinstance(obj[0], BaseModel):
        print(f"{prefix}{connector}{name or 'list'}: ({len(obj)} шт.)")
        for i, item in enumerate(obj):
            pretty_print(item, indent + 1, name=f"[{i}]")
    else:
        print(f"{prefix}{connector}{name or 'value'}: {obj}")

---
# Задача 1: как параметры генерации влияют на творческий текст (0.5 балла)

Выберите любимый фильм или книгу. Напишите промпт, который просит LLM придумать **альтернативную концовку** одним абзацем (4–6 предложений). Затем посмотрите, как на результат влияют `temperature` и `top_p` — **каждый параметр по отдельности**.

**Что сделать:**
1. Задайте переменную `my_movie_or_book`
2. Напишите промпт с альтернативной концовкой. Промпт должен:
   - ссылаться на фильм/книгу через переменную
   - просить ровно один абзац (4–6 предложений)
   - просить концовку креативную, но верную характерам героев и темам оригинала
3. Запустите промпт с разными параметрами и сравните результаты

**Важно:** когда задан только `temperature`, модель использует `top_p` по умолчанию (обычно 1.0). Когда задан только `top_p` — `temperature` по умолчанию (зависит от провайдера, обычно 0.7–1.0).

Используем `MODEL` — модель без рассуждений.

In [13]:
# TODO: выберите фильм/книгу и напишите промпт
my_movie_or_book = "Револьвер"  # <-- поменяйте!

# TODO: ваш промпт. Используйте переменную my_movie_or_book
prompt = f"""
Придумай альтернативную концовку для фильма "{my_movie_or_book}".

Напиши ровно один абзац из 4-6 предложений.
Концовка должна заметно отличаться от оригинальной и быть достаточно креативной,
но при этом сохранить характер главного героя, атмосферу фильма и его основные темы.
Не меняй героев так, чтобы их поведение противоречило тому, какими они показаны в оригинале.
Не пересказывай оригинальную концовку - предложи новый вариант развития событий.
"""

### Часть А — temperature (top_p по умолчанию)

Запустите промпт с 4 значениями temperature. Внимательно прочитайте каждый ответ и оцените его.

In [15]:
temperature_values = [0.1, 0.6, 1.0, 1.5]

for temp in temperature_values:
    print(f"=== temperature = {temp} ===\n")
    result = call_openrouter(MODEL, prompt, temperature=temp)
    # result = call_mistral("ministral-14b-latest", prompt, temperature=temp)
    print(result)
    print("\n" + "=" * 60 + "\n")

=== temperature = 0.1 ===

В альтернативной концовке фильма "Револьвер" Джейк Грин, осознав, что его истинный враг — это его собственное эго, решает использовать свои навыки и опыт, чтобы помочь другим освободиться от их внутренних демонов. Вместо того чтобы продолжать игру в месть и манипуляции, он открывает центр психологической помощи для бывших заключенных, где учит их преодолевать страхи и находить внутреннюю свободу. В финальной сцене мы видим, как Джейк, окруженный благодарными людьми, впервые за долгое время искренне улыбается, понимая, что настоящая победа — это победа над собой. Атмосфера фильма сохраняется через напряженные диалоги и внутренние монологи, но теперь они ведут к свету и надежде, а не к разрушению. Таким образом, фильм завершает свою историю на позитивной ноте, подчеркивая важность внутренней трансформации и силы прощения.


=== temperature = 0.6 ===

В альтернативной концовке фильма "Револьвер" Джейк Грин, осознав, что его враг — это не внешние силы, а его собс

**TODO:** заполните таблицу (1 — минимум, 5 — максимум):

| temperature | Креативность (1–5) | Связность (1–5) | Верность оригиналу (1–5) |
|---|---|---|---|
| 0.1 | | | |
| 0.6 | | | |
| 1.0 | | | |
| 1.5 | | | |

**DONE:**
| temperature | Креативность (1–5) | Связность (1–5) | Верность оригиналу (1–5) |
|---:|---:|---:|---:|
| **0.1** | **3/5** | **5/5** | **4/5** |
| **0.6** | **4/5** | **5/5** | **4/5** |
| **1.0** | **4/5** | **4/5** | **4/5** |
| **1.5** | **1/5** | **1/5** | **1/5** |

### Часть Б — top_p (temperature по умолчанию)

Запустите тот же промпт с 4 значениями top_p.

In [16]:
top_p_values = [0.5, 0.75, 0.92, 1.0]

for tp in top_p_values:
    print(f"=== top_p = {tp} ===\n")
    result = call_openrouter(MODEL, prompt, top_p=tp)
    # result = call_mistral("ministral-14b-latest", prompt, top_p=tp)
    print(result)
    print("\n" + "=" * 60 + "\n")

=== top_p = 0.5 ===

В альтернативной концовке фильма "Револьвер" Джейк Грин, осознав, что его истинный враг — это его собственное эго, решает использовать свои новые знания для создания тайной организации, цель которой — освобождение людей от власти их внутренних демонов. Вместо того чтобы мстить Мака, он предлагает ему присоединиться к его миссии, объясняя, что настоящая свобода заключается в преодолении своих страхов и иллюзий. Мака, пораженный неожиданной мудростью Джейка, соглашается, и они вместе начинают работать над тем, чтобы разоблачать и разрушать криминальные империи изнутри, используя психологические методы. В финальной сцене Джейк и Мака стоят на крыше небоскреба, наблюдая за городом, и Джейк говорит: "Мы — не те, кем нас делают наши страхи, мы — те, кем решаем стать". Атмосфера фильма сохраняется, но теперь в ней появляется надежда на внутреннее освобождение и возможность изменить мир к лучшему.


=== top_p = 0.75 ===

В альтернативной концовке "Револьвера" Джейк Грин, о

**TODO:** заполните таблицу:

| top_p | Креативность (1–5) | Связность (1–5) | Верность оригиналу (1–5) |
|---|---|---|---|
| 0.5 | | | |
| 0.75 | | | |
| 0.92 | | | |
| 1.0 | | | |

**DONE:**
| top_p | Креативность (1–5) | Связность (1–5) | Верность оригиналу (1–5) |
|---:|---:|---:|---:|
| **0.5** | **4/5** | **5/5** | **4/5** |
| **0.75** | **4/5** | **5/5** | **4/5** |
| **0.92** | **4/5** | **4/5** | **4/5** |
| **1.0** | **5/5** | **4/5** | **3/5** |

### Вывод

**TODO:** 3–4 предложения:
- Чем влияние temperature отличается от влияния top_p?
- С какого значения качество начало падать для каждого параметра?
- Какую одну настройку вы бы выбрали для этой задачи и почему?

*Ваш ответ:*
1. temperature сильнее влияет на случайность выбора токенов, те при росте значения ответы становятся более разнообразными, но при слишком высокой температуре связность может резко ухудшиться.
2. top_p управляет шириной множества допустимых токенов, поэтому рост top_p в моём эксперименте тоже увеличивал разнообразие, но делал это более плавно.
3. Для temperature качество начало заметно падать уже около 1.0, а при 1.5 генерация полностью потеряла связность;
4. для top_p ухудшение было заметно только ближе к 1.0, в основном по вайбу оригинала.
5. Для этой задачи я бы выбрал temperature=0.6, так как она дала лучший баланс между креативностью, связностью и схожестью оригиналу.


---
# Задача 2А: промпт-инжиниринг — извлечение данных из тикетов поддержки (1.5 балла)

Нужно извлечь структурированные данные из переписки клиента со службой поддержки. У правил извлечения **свои определения**, которые не всегда совпадают со здравым смыслом, — как в задаче про ипотеку в семинаре. Чтобы модель их соблюдала, нужен промпт-инжиниринг.

### Правила извлечения

- **`customer_name`**: имя клиента (строка)
- **`product_name`**: продукт, о котором идёт речь (строка)
- **`issue_type`**: одно из `billing`, `technical`, `feature_request`, `complaint`
- **`affected_feature`**: конкретная функция, которая сломана или обсуждается, или `null`
- **`workaround_exists`**: `true`, ТОЛЬКО если **агент поддержки подтвердил** обходной путь. Если клиент пишет «я попробовал X, и сработало» — это НЕ считается, агент это не подтвердил.
- **`sentiment_shift`**: сравнить ТОЛЬКО **первое и последнее сообщения клиента**. Одно из: `improved`, `worsened`, `stable`
- **`requires_followup`**: `true`, ТОЛЬКО если агент явно пообещал вернуться с ответом («я вернусь к вам»). Фразы вроде «напишите нам, если повторится» НЕ считаются — это перекладывает инициативу на клиента.
- **`promises_made`**: число конкретных обязательств агента. Правила:
  - обещание должно содержать **конкретное действие**: «я заведу тикет», «я отправлю письмо» — каждое считается за 1
  - «мы разберёмся» / «мы посмотрим» — НЕ считается (слишком расплывчато)
  - «я уточню у разработчиков и обновлю тикет» — считается за **2** (два разных действия)
  - повтор того же обещания — НЕ считается повторно
- **`resolution_provided`**: `true`, ТОЛЬКО если агент действительно **решил** проблему в ходе разговора, а не пообещал над ней поработать

In [1]:
!pip install -q openai==3.19.0

import os
import json
import re
from functools import lru_cache, partial

from kaggle_secrets import UserSecretsClient
from openai import OpenAI


# ---------- API KEY ----------

user_secrets = UserSecretsClient()

OPENROUTER_API_KEY = user_secrets.get_secret("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("❌ OPENROUTER_API_KEY не найден в Kaggle Secrets")

print("✅ OpenRouter API key loaded")


# ---------- CLIENT ----------

@lru_cache(maxsize=None)
def get_client():
    return OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        timeout=300
    )


# ---------- LLM CALL ----------

def build_messages(
    prompt: str,
    system_prompt: str | None = None
) -> list[dict]:

    messages = []

    if system_prompt:
        messages.append({
            "role": "system",
            "content": system_prompt
        })

    messages.append({
        "role": "user",
        "content": prompt
    })

    return messages


def chat(
    model: str,
    prompt: str | None = None,
    system_prompt: str | None = None,
    messages: list[dict] | None = None,
    **params
):
    client = get_client()

    if messages is None:
        messages = build_messages(
            prompt=prompt,
            system_prompt=system_prompt
        )

    return client.chat.completions.create(
        model=model,
        messages=messages,
        **params
    )


def call_openrouter(
    model: str,
    prompt: str | None = None,
    system_prompt: str | None = None,
    **params
):
    response = chat(
        model=model,
        prompt=prompt,
        system_prompt=system_prompt,
        **params
    )

    return response.choices[0].message.content


# ---------- JSON HELPERS ----------

def get_json_from_result(s: str, is_list: bool = False):
    pattern = re.compile(
        r"\[.*\]" if is_list else r"\{.*\}",
        re.DOTALL
    )

    json_match = pattern.search(s or "")

    if json_match is None:
        return None

    try:
        return json.loads(
            json_match.group(0),
            strict=False
        )
    except json.JSONDecodeError as e:
        print(f"❌ Не удалось разобрать JSON: {e}")
        return None


def flatten(d: dict, prefix: str = "") -> dict:
    items = {}

    for k, v in d.items():

        key = f"{prefix}.{k}" if prefix else k

        if isinstance(v, dict):
            items.update(flatten(v, key))
        else:
            items[key] = v

    return items


def values_match(expected, got) -> bool:

    if isinstance(expected, set):
        return any(
            values_match(e, got)
            for e in expected
        )

    if isinstance(expected, str) and isinstance(got, str):
        return expected.lower().strip() == got.lower().strip()

    return expected == got


def compare_json(
    result,
    expected: dict,
    skip_fields=None
) -> dict:

    skip_fields = skip_fields or []

    if isinstance(result, str):
        result = get_json_from_result(result)

    if result is None:
        print("❌ Не удалось достать JSON из ответа")
        return {"_": "нет JSON"}

    flat_result = flatten(result)
    flat_expected = flatten(expected)

    correct = []
    wrong = {}
    skipped = []

    for key, expected_val in flat_expected.items():

        if any(skip in key for skip in skip_fields):
            skipped.append(
                (key, flat_result.get(key, "НЕТ ПОЛЯ"))
            )
            continue

        got = flat_result.get(
            key,
            "НЕТ ПОЛЯ"
        )

        if values_match(expected_val, got):
            correct.append((key, got))
        else:
            wrong[key] = (
                expected_val,
                got
            )

    print(
        f"Верно: "
        f"{len(correct)}/"
        f"{len(correct) + len(wrong)}"
    )

    for key, val in correct:
        print(f"  ✅ {key}: {val!r}")

    for key, (exp, got) in wrong.items():
        print(
            f"  ❌ {key}: "
            f"ожидалось {exp!r}, "
            f"получено {got!r}"
        )

    for key, val in skipped:
        print(
            f"  👀 {key}: {val!r} "
            f"← проверьте вручную"
        )

    return wrong


print("✅ Setup complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 58.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.0/345.0 kB 18.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.85.7 requires openai<3.0.0,>=2.20.0, but you have openai 3.19.0 which is incompatible.
✅ OpenRouter API key loaded
✅ Setup complete


In [2]:
transcript_a = """
Артём: Здравствуйте, это Артём из финансового отдела. Меня попросила написать Елена — у неё проблемы с модулем отчётов в AnalyticsHub Pro. Графики выручки уже около недели показывают неправильные цифры.
Агент: Здравствуйте, Артём! Жаль это слышать. Уточните, пожалуйста: владелец аккаунта — вы или Елена?
Артём: Елена Смирнова — наш финансовый директор и владелец аккаунта. Я просто каждый день работаю с системой, поэтому она попросила меня разобраться.
Агент: Понял. То есть графики выручки в модуле отчётов показывают неверные данные — верно?
Артём: Да, и не только графики. Я заметил, что в выгруженных PDF-отчётах тоже неправильные итоги. Но честно говоря, графики важнее — Елена показывает их совету директоров.
Агент: Ясно. Я уже посмотрел: в прошлый вторник у нас был сбой в конвейере данных, который затронул часть Enterprise-аккаунтов. Сейчас проверю... да, ваш аккаунт тоже пострадал. Хорошая новость: вчера мы выкатили исправление.
Артём: Правда? Я сегодня утром проверял — цифры всё ещё выглядели странно.
Агент: Хм, исправление может применяться до 24 часов. Попробуйте, пожалуйста, сделать жёсткое обновление дашборда — это часто подтягивает свежие данные.
Артём: Сейчас попробую... Да, после жёсткого обновления графики правильные. Но насчёт PDF-выгрузок не уверен.
Агент: Отлично, что графики заработали! По PDF я создам тикет для команды экспорта, чтобы они проверили, что выгрузки берут данные из исправленного конвейера. И ещё передам вопрос нашей команде QA, чтобы они полностью проверили данные вашего аккаунта.
Артём: Хорошо. И ещё одно: мы уже несколько месяцев просим выгрузку в Excel, а не только в PDF. Есть новости?
Агент: Пока новостей нет, но я уточню у продуктовой команды и вернусь к вам с ответом. А пока я продлил вам пробный период дополнения Advanced Analytics на 30 дней — в качестве компенсации за неудобства.
Артём: Ладно, справедливо. Надеюсь, с PDF быстро разберутся — у нас скоро закрытие месяца.
"""

# Правильный ответ по нашим правилам:
expected_a = {
    "customer_name": "Елена Смирнова",       # владелец аккаунта — Елена; Артём пишет от её имени
    "product_name": "AnalyticsHub Pro",
    "issue_type": "technical",               # сбой конвейера данных — техническая проблема, а не жалоба
    "affected_feature": "модуль отчётов",    # и графики, и PDF — части модуля отчётов
    "workaround_exists": True,               # агент предложил жёсткое обновление, клиент подтвердил, агент: «Отлично, что графики заработали!»
    "sentiment_shift": "stable",             # первое сообщение — нейтральное описание проблемы, последнее — нейтральное с лёгкой тревогой
    "requires_followup": True,               # «уточню у продуктовой команды и вернусь к вам с ответом»
    "promises_made": 3,                      # 1) тикет для команды экспорта 2) передать QA 3) уточнить у продуктовой команды и вернуться;
                                             # продление пробного периода уже сделано — это не обещание
    "resolution_provided": False,            # графики починили, но PDF всё ещё сломаны — частичное решение ≠ решение
}

### Базовый промпт vs ваш промпт

Ниже — **базовый промпт**: он просто перечисляет правила извлечения. Инструкции идут в `system_prompt`, переписка — в `prompt`, как в семинаре. Запустите его и посмотрите, какие поля он извлекает неверно.

Затем напишите **свой промпт**, используя любые техники из семинара: Chain-of-Thought с разобранным примером, Few-shot, Self-Consistency, уточнение правил — или их комбинацию. Цель: все поля совпадают с `expected_a`.

Поле `affected_feature` сравнивается вручную (👀): модель может назвать модуль немного иначе, и это нормально.

In [5]:
MODEL = "openai/gpt-4o-mini"

In [6]:
# Базовый промпт — правила из описания выше, без примеров и уточнений
baseline_system = """Извлеки из переписки со службой поддержки следующие поля и верни JSON:
- customer_name: имя клиента
- product_name: продукт, о котором идёт речь
- issue_type: одно из billing, technical, feature_request, complaint
- affected_feature: конкретная функция, которая сломана или обсуждается, или null
- workaround_exists: true, ТОЛЬКО если агент поддержки подтвердил обходной путь. Если клиент пишет «я попробовал X, и сработало» — это НЕ считается.
- sentiment_shift: сравни ТОЛЬКО первое и последнее сообщения клиента. Одно из improved, worsened, stable
- requires_followup: true, ТОЛЬКО если агент явно пообещал вернуться с ответом («я вернусь к вам»). «Напишите нам, если повторится» НЕ считается.
- promises_made: число конкретных обязательств агента. Обещание должно содержать конкретное действие. «Мы разберёмся» НЕ считается. «Я уточню у разработчиков и обновлю тикет» считается за 2. Повтор того же обещания НЕ считается повторно.
- resolution_provided: true, ТОЛЬКО если агент действительно решил проблему, а не пообещал над ней поработать."""

baseline_result = call_openrouter(MODEL, transcript_a, system_prompt=baseline_system, temperature=0)
# baseline_result = call_mistral("ministral-14b-latest", transcript_a, system_prompt=baseline_system, temperature=0)
print(baseline_result, "\n")
compare_json(baseline_result, expected_a, skip_fields=["affected_feature"]);

```json
{
  "customer_name": "Елена Смирнова",
  "product_name": "AnalyticsHub Pro",
  "issue_type": "technical",
  "affected_feature": "PDF-отчёты",
  "workaround_exists": true,
  "sentiment_shift": "improved",
  "requires_followup": true,
  "promises_made": 2,
  "resolution_provided": true
}
``` 

Верно: 5/8
  ✅ customer_name: 'Елена Смирнова'
  ✅ product_name: 'AnalyticsHub Pro'
  ✅ issue_type: 'technical'
  ✅ workaround_exists: True
  ✅ requires_followup: True
  ❌ sentiment_shift: ожидалось 'stable', получено 'improved'
  ❌ promises_made: ожидалось 3, получено 2
  ❌ resolution_provided: ожидалось False, получено True
  👀 affected_feature: 'PDF-отчёты' ← проверьте вручную


In [7]:
# TODO: напишите свой промпт, используя техники из семинара
# (CoT с разобранным примером, Few-shot, Self-Consistency, уточнение правил — или комбинацию).
# Цель: ВСЕ поля совпадают с expected_a.
# Если просите модель рассуждать, попросите вернуть JSON в конце — compare_json достанет его из текста.

my_system_prompt = """
Ты анализируешь переписку клиента со службой поддержки и извлекаешь
структурированные данные строго по правилам ниже.

ВАЖНО: не используй обычный здравый смысл вместо этих правил.
Перед формированием ответа мысленно проверь каждое поле отдельно.

Правила:

1. customer_name
Имя клиента / владельца аккаунта, от имени которого ведётся обращение.
Если пишет другой сотрудник от имени владельца аккаунта, укажи владельца.

2. product_name
Название основного продукта, с которым связана проблема.

3. issue_type
Ровно одно значение:
- billing
- technical
- feature_request
- complaint

Если причиной является технический сбой системы, выбирай technical,
даже если клиент недоволен.

4. affected_feature
Укажи наиболее общий компонент продукта, который объединяет затронутые функции.
Если несколько симптомов относятся к одному модулю, укажи этот модуль.

5. workaround_exists
true ТОЛЬКО если:
- агент предложил обходной путь;
- клиент попробовал его;
- агент подтвердил, что он сработал.

Одного сообщения клиента "я попробовал, получилось" недостаточно,
если агент это не подтвердил.

6. sentiment_shift
Сравнивай ТОЛЬКО первое и последнее сообщения клиента.
Игнорируй эмоциональные изменения в середине разговора.

- improved — последнее сообщение явно позитивнее первого;
- worsened — явно негативнее;
- stable — эмоциональный тон существенно не изменился.

Если в последнем сообщении проблема всё ещё беспокоит клиента,
но тон остаётся нейтральным или слегка тревожным, это stable,
а не improved.

7. requires_followup
true ТОЛЬКО если агент сам явно обещает вернуться с ответом,
например: "я уточню и вернусь к вам".

Просьба клиенту самостоятельно написать позже НЕ считается follow-up.

8. promises_made
Посчитай все УНИКАЛЬНЫЕ конкретные будущие действия агента.

Каждое отдельное действие = 1.

Например:
"Я создам тикет и передам вопрос QA" = 2 обещания.

"Я уточню у продуктовой команды и вернусь к вам с ответом"
считай как ОДНО составное обязательство по получению ответа и возврату клиенту.

Не считай:
- расплывчатые фразы вроде "разберёмся";
- уже выполненные действия;
- повтор одного и того же обещания.

9. resolution_provided
true ТОЛЬКО если ВСЯ основная проблема обращения полностью решена
во время разговора.

Если исправлена только одна часть проблемы, а другая часть всё ещё
требует тикета, проверки или дальнейшей работы, ставь false.

Перед ответом отдельно проверь:
- решены ли ВСЕ затронутые части проблемы;
- сколько именно будущих конкретных действий пообещал агент;
- действительно ли тон последнего сообщения клиента лучше первого.

Верни ТОЛЬКО валидный JSON без Markdown и пояснений:

{
  "customer_name": "...",
  "product_name": "...",
  "issue_type": "...",
  "affected_feature": "...",
  "workaround_exists": true,
  "sentiment_shift": "...",
  "requires_followup": true,
  "promises_made": 0,
  "resolution_provided": false
}
"""

my_result = call_openrouter(
    MODEL,
    transcript_a,
    system_prompt=my_system_prompt,
    temperature=0
)# my_result = call_mistral("ministral-14b-latest", transcript_a, system_prompt=my_system_prompt, temperature=0)
print(my_result, "\n")

compare_json(
    my_result,
    expected_a,
    skip_fields=["affected_feature"]
)

{
  "customer_name": "Елена Смирнова",
  "product_name": "AnalyticsHub Pro",
  "issue_type": "technical",
  "affected_feature": "модуль отчётов",
  "workaround_exists": true,
  "sentiment_shift": "stable",
  "requires_followup": true,
  "promises_made": 2,
  "resolution_provided": false
} 

Верно: 7/8
  ✅ customer_name: 'Елена Смирнова'
  ✅ product_name: 'AnalyticsHub Pro'
  ✅ issue_type: 'technical'
  ✅ workaround_exists: True
  ✅ sentiment_shift: 'stable'
  ✅ requires_followup: True
  ✅ resolution_provided: False
  ❌ promises_made: ожидалось 3, получено 2
  👀 affected_feature: 'модуль отчётов' ← проверьте вручную


{'promises_made': (3, 2)}

**TODO:** 2–3 предложения: на каких полях ошибался базовый промпт, какую технику вы применили и почему она помогла?

*Ваш ответ:*

Базовый промпт ошибался в sentiment_shift, promises_made и resolution_provided: модель слишком свободно интерпретировала изменение настроения, частичное решение проблемы и количество обещаний агента. Я добавил пошаговую декомпозицию и уточнил пограничные правила с конкретными примерами, благодаря чему удалось исправить sentiment_shift и resolution_provided; promises_made всё ещё требует дополнительного уточнения, так как модель посчитала 2 обещания вместо 3.

Чтобы добить до 8/8, я бы добавил в prompt буквально такой few-shot:
Пример подсчёта promises_made:
- "Я создам тикет для команды экспорта" = 1
- "Я передам вопрос QA" = 1
- "Я уточню у продуктовой команды и вернусь с ответом" = 1

Итого: promises_made = 3.

то есть получается:

sentiment_shift
- не "как в целом стало лучше",
- а сравнить только первое и последнее сообщения

resolution_provided
- не "часть починилась",
- а решена ли вся проблема

promises_made
- не просто посчитать фразы,
- а применить специальное правило подсчёта обещаний

---
# Задача 2Б: reasoning-модели и их параметры (1 балл)

В семинаре мы увидели, что reasoning-модели не отменяют чёткой постановки задачи. Проверим это на тикетах — и заодно разберёмся с параметрами reasoning-моделей:

- **`reasoning_effort`** (`low` / `medium` / `high`) — сколько модели рассуждать. Это *пожелание*: не все модели и не все провайдеры его учитывают;
- **`max_tokens`** — у reasoning-модели лимит расходуется и на рассуждения, и на ответ;
- **рассуждения модели** (`reasoning_content`) — по ним видно, *почему* модель ошиблась.

Функция `evaluate_ticket` делает один запрос и печатает строку с результатом: ошибки, число токенов в ответе (вместе с рассуждениями), время и `finish_reason`. Возвращает сообщение модели — из него можно достать рассуждения.

⏱ Reasoning-модели отвечают медленно: `gemma-4-31b` — до 1–2 минут на запрос, `qwen3.6-35b-a3b` — до минуты и 3–5 тысяч токенов. `gpt-oss-120b` — обычно 10–20 секунд.

In [14]:
# def get_reasoning(message) -> str:
#     # Поле с рассуждениями нестандартное: у разных провайдеров оно называется по-разному
#     extra = message.model_extra or {}
#     return getattr(message, "reasoning_content", None) or extra.get("reasoning_content") or extra.get("reasoning") or ""


# def evaluate_ticket(model: str, system_prompt: str, transcript: str = transcript_a, expected: dict = expected_a,
#                     skip_fields=("affected_feature",), **params):
#     # Один запрос: сравниваем ответ с правильным и печатаем ошибки, токены, время и finish_reason
#     start = time.perf_counter()
#     response = chat("yandex", model, transcript, system_prompt=system_prompt, **params)
#     elapsed = time.perf_counter() - start

#     message = response.choices[0].message
#     result = get_json_from_result(message.content)
#     errors = {k: result.get(k, "НЕТ ПОЛЯ") for k, v in expected.items()
#               if k not in skip_fields and not values_match(v, result.get(k, "НЕТ ПОЛЯ"))} if result else None
#     status = "✅ всё верно" if errors == {} else f"❌ ошибки: {errors if errors else 'нет JSON'}"
#     params_str = ", ".join(f"{k}={v}" for k, v in params.items())
#     print(f"{model}({params_str}): {status}\n"
#           f"    токенов в ответе: {response.usage.completion_tokens}, время: {elapsed:.0f} с, "
#           f"finish_reason: {response.choices[0].finish_reason}")
#     return message
import time


def get_reasoning(message) -> str:
    """
    Достаём reasoning из ответа OpenRouter.
    У разных моделей поле может отличаться.
    """
    extra = message.model_extra or {}

    reasoning = (
        getattr(message, "reasoning", None)
        or getattr(message, "reasoning_content", None)
        or extra.get("reasoning")
        or extra.get("reasoning_content")
    )

    if reasoning:
        return str(reasoning)

    # Некоторые OpenRouter-модели возвращают reasoning_details
    details = (
        getattr(message, "reasoning_details", None)
        or extra.get("reasoning_details")
    )

    if details:
        return str(details)

    return ""


def evaluate_ticket(
    model: str,
    system_prompt: str,
    transcript: str = transcript_a,
    expected: dict = expected_a,
    skip_fields=("affected_feature",),
    **params
):
    start = time.perf_counter()

    # Просим OpenRouter вернуть reasoning, если модель его поддерживает
    extra_body = params.pop("extra_body", {})
    extra_body["include_reasoning"] = True

    response = chat(
        model=model,
        prompt=transcript,
        system_prompt=system_prompt,
        extra_body=extra_body,
        **params
    )

    elapsed = time.perf_counter() - start

    message = response.choices[0].message

    result = get_json_from_result(message.content)

    if result:
        errors = {
            k: result.get(k, "НЕТ ПОЛЯ")
            for k, v in expected.items()
            if (
                k not in skip_fields
                and not values_match(
                    v,
                    result.get(k, "НЕТ ПОЛЯ")
                )
            )
        }
    else:
        errors = None

    status = (
        "✅ всё верно"
        if errors == {}
        else f"❌ ошибки: {errors if errors else 'нет JSON'}"
    )

    params_str = ", ".join(
        f"{k}={v}"
        for k, v in params.items()
    )

    completion_tokens = (
        response.usage.completion_tokens
        if response.usage
        else "?"
    )

    print(
        f"{model} ({params_str}): {status}\n"
        f"    токенов в ответе: {completion_tokens}, "
        f"время: {elapsed:.1f} с, "
        f"finish_reason: {response.choices[0].finish_reason}"
    )

    return message

### Б.1 Помогают ли рассуждения без промпт-инжиниринга?

Запустите базовый промпт и ваш промпт из 2А на моделях с рассуждениями и без. Поведение вероятностное — если результат неожиданный, запустите ячейку ещё раз.

In [10]:
MODEL = "openai/gpt-4o"
THINKING_MODEL = "openai/gpt-4o"
REASONING_MODEL = "openai/gpt-4o"
EFFORT_MODEL = "openai/gpt-4o"

In [15]:
# compared_models = [MODEL, THINKING_MODEL, EFFORT_MODEL]  # можно добавить REASONING_MODEL — дольше и дороже

# for name, system_prompt in [("Базовый промпт", baseline_system), ("Ваш промпт", my_system_prompt)]:
#     print(f"=== {name} ===")
#     for model in compared_models:
#         evaluate_ticket(model, system_prompt)
#     print()
compared_models = [
    MODEL,
    THINKING_MODEL,
    EFFORT_MODEL
]

for name, system_prompt in [
    ("Базовый промпт", baseline_system),
    ("Ваш промпт", my_system_prompt)
]:
    print(f"\n=== {name} ===")

    for model in compared_models:
        evaluate_ticket(
            model,
            system_prompt
        )

    print()


=== Базовый промпт ===
openai/gpt-4o (): ❌ ошибки: {'sentiment_shift': 'improved', 'resolution_provided': True}
    токенов в ответе: 99, время: 1.3 с, finish_reason: stop
openai/gpt-4o (): ❌ ошибки: {'sentiment_shift': 'improved', 'resolution_provided': True}
    токенов в ответе: 93, время: 1.6 с, finish_reason: stop
openai/gpt-4o (): ❌ ошибки: {'sentiment_shift': 'worsened', 'resolution_provided': True}
    токенов в ответе: 94, время: 1.0 с, finish_reason: stop


=== Ваш промпт ===
openai/gpt-4o (): ❌ ошибки: {'promises_made': 2}
    токенов в ответе: 90, время: 1.1 с, finish_reason: stop
openai/gpt-4o (): ✅ всё верно
    токенов в ответе: 86, время: 1.2 с, finish_reason: stop
openai/gpt-4o (): ✅ всё верно
    токенов в ответе: 86, время: 0.9 с, finish_reason: stop



**TODO:** заполните таблицу:

| Модель | Промпт | Число ошибок | Токенов в ответе | Время, с |
|---|---|---|---|---|
| qwen3.6-35b-a3b-noreason | базовый | | | |
| qwen3.6-35b-a3b | базовый | | | |
| gpt-oss-120b | базовый | | | |
| qwen3.6-35b-a3b-noreason | ваш | | | |
| qwen3.6-35b-a3b | ваш | | | |
| gpt-oss-120b | ваш | | | |

**DONE:**
| Реальный запуск | Промпт | Ошибок | Токенов | Время |
|---|---|---:|---:|---:|
| GPT-4o run 1 | базовый | 2 | 99 | 1.3 |
| GPT-4o run 2 | базовый | 2 | 93 | 1.6 |
| GPT-4o run 3 | базовый | 2 | 94 | 1.0 |
| GPT-4o run 1 | ваш | 1 | 90 | 1.1 |
| GPT-4o run 2 | ваш | 0 | 86 | 1.2 |
| GPT-4o run 3 | ваш | 0 | 86 | 0.9 |

### Б.2 Параметр `reasoning_effort`

Запустите `EFFORT_MODEL` с разными значениями `reasoning_effort` на базовом промпте и на вашем.

In [16]:
EFFORT_MODEL = "openai/gpt-oss-120b"

In [17]:
for name, system_prompt in [("Базовый промпт", baseline_system), ("Ваш промпт", my_system_prompt)]:
    print(f"=== {name} ===")
    for effort in ["low", "medium", "high"]:
        evaluate_ticket(EFFORT_MODEL, system_prompt, reasoning_effort=effort)
    print()

=== Базовый промпт ===
openai/gpt-oss-120b (reasoning_effort=low): ❌ ошибки: {'customer_name': 'Артём', 'sentiment_shift': 'improved', 'promises_made': 4}
    токенов в ответе: 449, время: 26.8 с, finish_reason: stop
openai/gpt-oss-120b (reasoning_effort=medium): ❌ ошибки: {'customer_name': 'Артём', 'sentiment_shift': 'improved', 'promises_made': 5, 'resolution_provided': True}
    токенов в ответе: 1238, время: 36.9 с, finish_reason: stop
openai/gpt-oss-120b (reasoning_effort=high): ❌ ошибки: {'customer_name': 'Артём', 'workaround_exists': False, 'sentiment_shift': 'improved', 'promises_made': 4}
    токенов в ответе: 4440, время: 41.5 с, finish_reason: stop

=== Ваш промпт ===
openai/gpt-oss-120b (reasoning_effort=low): ✅ всё верно
    токенов в ответе: 338, время: 0.9 с, finish_reason: stop
openai/gpt-oss-120b (reasoning_effort=medium): ❌ ошибки: {'sentiment_shift': 'improved'}
    токенов в ответе: 706, время: 56.0 с, finish_reason: stop
openai/gpt-oss-120b (reasoning_effort=high):

**TODO:** заполните таблицу:

| reasoning_effort | Промпт | Число ошибок | Токенов в ответе | Время, с |
|---|---|---|---|---|
| low | базовый | | | |
| medium | базовый | | | |
| high | базовый | | | |
| low | ваш | | | |
| medium | ваш | | | |
| high | ваш | | | |

**DONE:**

| reasoning_effort | Промпт  | Число ошибок | Токенов в ответе | Время, с |
|---|---|---:|---:|---:|
| low    | базовый | 3 | 449  | 26.8 |
| medium | базовый | 4 | 1238 | 36.9 |
| high   | базовый | 4 | 4440 | 41.5 |
| low    | ваш     | 0 | 338  | 0.9  |
| medium | ваш     | 1 | 706  | 56.0 |
| high   | ваш     | 0 | 2679 | 73.1 |

*Дополнительно (по желанию):* передайте `reasoning_effort="low"` и `"high"` в `THINKING_MODEL` или `REASONING_MODEL`. Меняется ли что-нибудь?

### Б.3 Почему модель ошиблась: читаем рассуждения

Запустите базовый промпт на `EFFORT_MODEL` и прочитайте рассуждения модели. Найдите место, где она неверно применила правило (например, решила, чьё имя писать в `customer_name`, или посчитала обещания). Если в этом запуске ошибок не было, запустите ячейку ещё раз.

In [20]:
message = evaluate_ticket(EFFORT_MODEL, baseline_system, reasoning_effort="high")

print("\n=== Рассуждения ===")
print(get_reasoning(message))
print("\n=== Ответ ===")
print(message.content)

openai/gpt-oss-120b (reasoning_effort=high): ❌ ошибки: {'customer_name': 'Артём', 'workaround_exists': False, 'sentiment_shift': 'improved', 'promises_made': 4}
    токенов в ответе: 4410, время: 40.7 с, finish_reason: stop

=== Рассуждения ===
We need to parse the conversation and produce JSON with the required fields. Let's restate the conversation.

Conversation: 
User: "Артём: ..." Actually, the conversation includes two speakers: "Артём" (client) and "Агент" (support agent). So we need to extract required fields.

### Fields:

- customer_name: имя клиента. The name of the client user is "Артём". But sometimes the client is speaking on behalf of "Елена Смирнова". The request: "customer_name: имя клиента". Usually refers to the name of the person who is communicating (the client). So likely "Артём". Could also include "Елена Смирнова". But the instruction says "имя клиента". The client is Artёm (the person writing). So we will set "customer_name": "Артём".

- product_name: продукт, 

**TODO:** процитируйте фрагмент рассуждений, где модель ошиблась. Какое правило она поняла не так, как задумано? Как бы вы переформулировали это правило?

*Ваш ответ:*

Модель неверно поняла несколько правил: например, для customer_name она решила, что клиент — это тот, кто пишет в чат, и выбрала Артёма, хотя по условию нужно было брать владельца аккаунта Елену Смирнову. Также она посчитала фразу "уточню у продуктовой команды и вернусь к вам с ответом" как два отдельных обещания, из-за чего получила promises_made = 4, хотя в этой задаче это должно считаться одним составным обязательством.

Я бы переформулировал правила так: customer_name - владелец аккаунта, даже если пишет другой сотрудник; promises_made - считать уникальные обязательства по смыслу, а последовательные действия в рамках одного обещания клиенту считать как одно, если они образуют одно целое.

### Б.4 Ловушка с `max_tokens`

Запустите ваш промпт на `EFFORT_MODEL` с маленьким лимитом `max_tokens`.

In [21]:
message = evaluate_ticket(EFFORT_MODEL, my_system_prompt, max_tokens=300)
print(f"\ncontent: {message.content!r}")
print(f"Длина рассуждений: {len(get_reasoning(message))} символов")

openai/gpt-oss-120b (max_tokens=300): ❌ ошибки: нет JSON
    токенов в ответе: 300, время: 2.3 с, finish_reason: length

content: None
Длина рассуждений: 1305 символов


### Вывод по задаче 2Б

**TODO:** 4–5 предложений:
- Исправили ли рассуждения ошибки базового промпта? Что помогло больше — reasoning или ваш промпт?
- Во сколько раз reasoning-модели дороже по токенам и медленнее? Когда это оправдано?
- Как `reasoning_effort` повлиял на токены, время и качество? Для каких моделей параметр работает?
- Что произошло при маленьком `max_tokens` и почему?

*Ваш ответ:*

Reasoning сам по себе не исправил ошибки базового промпта: даже при high модель продолжала ошибаться, тогда как уточнённый промпт дал правильный результат уже при low, поэтому в этом эксперименте качество инструкции оказалось важнее глубины рассуждений. Увеличение reasoning_effort резко повышало расход токенов: примерно с 338–449 на low до 2679–4440 на high, то есть примерно в 8–10 раз, а время ответа также заметно возрастало. При этом больше reasoning не гарантировало лучшее качество: medium иногда давал больше ошибок, чем low. Параметр reasoning_effort в нашем эксперименте работал для gpt-oss-120b; для других моделей его поддержка зависит от самой модели и провайдера. При max_tokens=300 весь лимит ушёл на рассуждение, finish_reason стал length, а финальный JSON вообще не успел сгенерироваться = reasoning и итоговый ответ делят общий "токен-бюджет".

---
# Задача 3: структурированный ответ с Pydantic — извлечение вакансии (1.5 балла)

Опишите вложенные Pydantic-модели для вакансии и извлеките из текста вакансии структурированные данные.

### Что сделать
1. Опишите Pydantic-модели (поля перечислены в заготовке ниже). **Типы полей выбираете вы**: где значение из фиксированного списка — подумайте, как ограничить его схемой (подсказка: `Literal`); где значения может не быть — `Optional`. Можно добавлять `Field(description=...)`.
2. Напишите системный промпт с правилами извлечения
3. Вызовите `call_neuraldeep(THINKING_MODEL, job_posting, system_prompt=..., response_format=JobPosting)`
4. Сравните результат с правильным ответом

### Правила извлечения
- `salary_min` / `salary_max` / `currency`: `null`, если зарплата «конкурентная» или не указана
- `min_years_experience`: МИНИМУМ из диапазона («3–5 лет» → 3)
- `company.size`: одно из `startup`, `mid-size`, `enterprise`
- `education_level`: одно из `high_school`, `bachelor`, `master`, `phd` или `null`, если не требуется
- `seniority`: одно из `junior`, `mid`, `senior`, `lead`
- `remote_policy`: одно из `remote`, `hybrid`, `onsite`. `hybrid` — только если есть регулярные обязательные дни в офисе; редкие выезды не делают работу гибридной
- `required_skills` vs `preferred_skills`: «обязательно» / «требуется» → required; «будет плюсом» / «бонус» → preferred
- `has_equity`: `true`, только если опционы/доля явно предлагаются **для этой роли** (не «опционы для senior-ролей», если вакансия уровня middle)

In [22]:
job_posting = """
Backend-разработчик — FinFlow (финтех-стартап, раунд B)

О нас: FinFlow — быстрорастущий финтех-стартап (120 сотрудников), который строит платёжную инфраструктуру
нового поколения. Офис — в Берлине, Германия.

Роль: ищем backend-разработчика уровня middle в команду Payments.
Вы будете проектировать и поддерживать API, которые обрабатывают миллионы транзакций в день.

Требования:
- 3–5 лет опыта в backend-разработке
- Уверенное знание Python и SQL (обязательно)
- Опыт работы с PostgreSQL и Redis (обязательно)
- Понимание событийно-ориентированной архитектуры
- Высшее образование (бакалавриат) в Computer Science или эквивалентный практический опыт
- Английский на уровне B2+ — рабочий язык команды

Будет плюсом:
- Опыт с Kubernetes и AWS
- Знание платёжных протоколов (ISO 8583, PCI-DSS)
- Вклад в open-source проекты

Мы предлагаем:
- Конкурентную зарплату (ориентируемся на топовые компании)
- Опционы для senior-ролей
- Удалённую работу: офис в Берлине есть, но работать можно из любой страны ЕС.
  Раз в квартал — командные выезды (по 3 дня).
- 30 дней отпуска, бюджет на обучение, абонемент в спортзал

Откликаться: careers@finflow.io
"""

In [27]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field, ConfigDict


class StrictBaseModel(BaseModel):
    # Запрещаем любые поля, которых нет в схеме
    model_config = ConfigDict(extra="forbid")


class Company(StrictBaseModel):
    name: str = Field(
        description="Название компании"
    )

    industry: str = Field(
        description="Отрасль компании"
    )

    size: Literal["startup", "mid-size", "enterprise"] = Field(
        description="Размер компании"
    )

    location: str = Field(
        description="Основная локация компании"
    )


class Compensation(StrictBaseModel):
    salary_min: Optional[int] = Field(
        description=(
            "Минимальная зарплата. "
            "Если зарплата не указана числом или названа конкурентной — null"
        )
    )

    salary_max: Optional[int] = Field(
        description=(
            "Максимальная зарплата. "
            "Если зарплата не указана числом или названа конкурентной — null"
        )
    )

    currency: Optional[str] = Field(
        description=(
            "Валюта зарплаты. "
            "Если числовая зарплата не указана — null"
        )
    )

    has_equity: bool = Field(
        description=(
            "true только если опционы или доля "
            "предлагаются именно для этой роли"
        )
    )

    benefits: List[str] = Field(
        description="Список льгот и бонусов"
    )


class Requirements(StrictBaseModel):
    min_years_experience: Optional[int] = Field(
        description=(
            "Минимум из указанного диапазона опыта. "
            "Например 3–5 лет -> 3"
        )
    )

    education_level: Optional[
        Literal["high_school", "bachelor", "master", "phd"]
    ] = Field(
        description=(
            "Требуемый уровень образования или null, "
            "если образование не требуется"
        )
    )

    required_skills: List[str] = Field(
        description="Обязательные навыки"
    )

    preferred_skills: List[str] = Field(
        description="Желательные навыки / будет плюсом"
    )

    languages: List[str] = Field(
        description="Требуемые языки и уровни владения"
    )


class RoleDetails(StrictBaseModel):
    title: str = Field(
        description="Название вакансии"
    )

    seniority: Literal["junior", "mid", "senior", "lead"] = Field(
        description="Уровень позиции"
    )

    department: str = Field(
        description="Команда или подразделение"
    )

    remote_policy: Literal["remote", "hybrid", "onsite"] = Field(
        description=(
            "remote — полностью удалённо; "
            "hybrid — регулярные обязательные дни в офисе; "
            "onsite — постоянная работа из офиса"
        )
    )

    responsibilities: List[str] = Field(
        description="Основные обязанности"
    )


class JobPosting(StrictBaseModel):
    company: Company
    compensation: Compensation
    requirements: Requirements
    role: RoleDetails

In [24]:
def call_openrouter_pydantic(
    model: str,
    prompt: str,
    system_prompt: str,
    response_model,
    **params
):
    client = get_client()

    schema = response_model.model_json_schema()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "strict": True,
                "schema": schema
            }
        },
        **params
    )

    content = response.choices[0].message.content

    # JSON -> валидированный Pydantic object
    return response_model.model_validate_json(content)

In [29]:
def pretty_print(obj, indent: int = 0, name: str | None = None):
    prefix = "  " * indent
    connector = "├── " if indent > 0 else ""

    if isinstance(obj, BaseModel):
        print(f"{prefix}{connector}{name or obj.__class__.__name__}:")
        for field_name in type(obj).model_fields:
            pretty_print(
                getattr(obj, field_name),
                indent + 1,
                name=field_name
            )

    elif isinstance(obj, list):
        print(f"{prefix}{connector}{name or 'list'}:")
        for i, item in enumerate(obj):
            pretty_print(
                item,
                indent + 1,
                name=f"[{i}]"
            )

    else:
        print(f"{prefix}{connector}{name or 'value'}: {obj}")

In [30]:
# TODO: напишите системный промпт и извлеките данные

job_system_prompt = """
Извлеки из текста вакансии структурированные данные строго по заданной схеме.

Не додумывай информацию, которой нет в вакансии.
Используй следующие правила:

1. company.size:
   - startup — если компания явно названа стартапом;
   - mid-size — средняя компания;
   - enterprise — крупная корпорация.

2. salary_min, salary_max и currency:
   если зарплата указана только как "конкурентная",
   "рыночная" или числовой диапазон отсутствует,
   вернуть null для всех трёх полей.

3. min_years_experience:
   если указан диапазон, например "3–5 лет",
   вернуть минимальное значение — 3.

4. education_level:
   допустимые значения:
   high_school, bachelor, master, phd.
   Если образование не требуется — null.
   "Бакалавриат" → bachelor.

5. seniority:
   допустимые значения:
   junior, mid, senior, lead.
   "middle" следует преобразовать в "mid".

6. remote_policy:
   - remote — если можно постоянно работать удалённо;
   - hybrid — только если есть регулярные обязательные дни в офисе;
   - onsite — если работа требует постоянного присутствия в офисе.
   Редкие командные выезды или встречи не делают работу hybrid.

7. required_skills:
   включай только навыки, которые явно обязательны
   или находятся в разделе требований как необходимые.

8. preferred_skills:
   включай навыки из разделов "будет плюсом",
   "желательно", "бонус" и аналогичных.

9. has_equity:
   true только если опционы или доля предлагаются именно для этой вакансии.
   Если написано, что опционы доступны только для другого уровня,
   например senior, а вакансия middle — вернуть false.

10. benefits:
    перечисли реально предлагаемые льготы и бонусы.

11. responsibilities:
    перечисли основные обязанности данной роли.

Верни данные строго согласно переданной JSON-схеме.
"""

result = call_openrouter_pydantic(THINKING_MODEL, job_posting, system_prompt=job_system_prompt, response_model=JobPosting)
# result = call_neuraldeep(MODEL, job_posting, system_prompt=job_system_prompt, response_format=JobPosting)  # быстрее
# result = call_mistral("ministral-14b-latest", job_posting, system_prompt=job_system_prompt, response_format=JobPosting)

pretty_print(result)

JobPosting:
  ├── company:
    ├── name: FinFlow
    ├── industry: Financial Technology
    ├── size: startup
    ├── location: Berlin, Germany
  ├── compensation:
    ├── salary_min: None
    ├── salary_max: None
    ├── currency: None
    ├── has_equity: False
    ├── benefits:
      ├── [0]: 30 дней отпуска
      ├── [1]: бюджет на обучение
      ├── [2]: абонемент в спортзал
  ├── requirements:
    ├── min_years_experience: 3
    ├── education_level: bachelor
    ├── required_skills:
      ├── [0]: Python
      ├── [1]: SQL
      ├── [2]: PostgreSQL
      ├── [3]: Redis
    ├── preferred_skills:
      ├── [0]: Kubernetes
      ├── [1]: AWS
      ├── [2]: платёжные протоколы (ISO 8583, PCI-DSS)
      ├── [3]: вклад в open-source проекты
    ├── languages:
      ├── [0]: English B2+
  ├── role:
    ├── title: Backend-разработчик
    ├── seniority: mid
    ├── department: Payments
    ├── remote_policy: remote
    ├── responsibilities:
      ├── [0]: Проектировать и поддерживать API д

In [32]:
# Проверка. Множество {...} — несколько допустимых вариантов
expected_posting = {
    "company": {"name": "FinFlow", "industry": {"fintech", "финтех"}, "size": "startup",
                "location": {"Берлин, Германия", "Berlin, Germany", "Берлин"}},
    "compensation": {
        "salary_min": None,
        "salary_max": None,
        "currency": None,
        "has_equity": False,
    },
    "requirements": {
        "min_years_experience": 3,
        "education_level": "bachelor",
    },
    "role": {
        "title": {"Backend-разработчик", "Backend Engineer", "Backend Developer"},
        "seniority": "mid",
        "department": "Payments",
        "remote_policy": "remote",
    },
}

# Поля-списки (навыки, льготы, обязанности) сравниваем вручную — порядок и формулировки могут отличаться
compare_json(result.model_dump(), expected_posting);
print()
print(f"required_skills:  {result.requirements.required_skills}")
print(f"preferred_skills: {result.requirements.preferred_skills}")
print(f"languages:        {result.requirements.languages}")
print(f"benefits:         {result.compensation.benefits}")

Верно: 13/14
  ✅ company.name: 'FinFlow'
  ✅ company.size: 'startup'
  ✅ company.location: 'Berlin, Germany'
  ✅ compensation.salary_min: None
  ✅ compensation.salary_max: None
  ✅ compensation.currency: None
  ✅ compensation.has_equity: False
  ✅ requirements.min_years_experience: 3
  ✅ requirements.education_level: 'bachelor'
  ✅ role.title: 'Backend-разработчик'
  ✅ role.seniority: 'mid'
  ✅ role.department: 'Payments'
  ✅ role.remote_policy: 'remote'
  ❌ company.industry: ожидалось {'финтех', 'fintech'}, получено 'Financial Technology'

required_skills:  ['Python', 'SQL', 'PostgreSQL', 'Redis']
preferred_skills: ['Kubernetes', 'AWS', 'платёжные протоколы (ISO 8583, PCI-DSS)', 'вклад в open-source проекты']
languages:        ['English B2+']
benefits:         ['30 дней отпуска', 'бюджет на обучение', 'абонемент в спортзал']


**TODO:** 2–3 предложения: какие поля получились неверно с первой попытки и что помогло их исправить — тип поля в схеме, `Field(description=...)` или правило в промпте?

*Ваш ответ:*

С первой попытки наиболее проблемными были поля, где требовалось строгое бизнес-правило: зарплата, has_equity, remote_policy, min_years_experience и категориальные значения вроде seniority. Их удалось стабилизировать сочетанием Literal/Optional, уточняющих Field(description=...) и явных правил в системном промпте. В итоговом запуске неверным осталось только company.industry: модель вернула Financial Technology вместо ожидаемого fintech, то есть здесь стоило дополнительно ограничить поле через Literal или явно потребовать нормализовать отрасль до fintech.

---
# Задача 4: Schema-Guided Reasoning — планировщик путешествий (2 балла)

Соберём планировщик путешествий на SGR. Пользователь даёт минимум информации: куда, на сколько дней, тип поездки, пара обязательных мест. Модель сама по своим знаниям находит достопримечательности, строит план по дням и проверяет его.

Реализуете все три паттерна из семинара:
1. **Каскад**: разобрать запрос → найти достопримечательности → построить план по дням
2. **Маршрутизация**: разные требования для активного, спокойного и культурного отдыха
3. **Цикл**: улучшать план, пока не выполнены все ограничения

Везде используем `THINKING_MODEL`. ⏱ Большие схемы заполняются долго: один вызов — 1–4 минуты. Для отладки схемы можно временно заменить модель на более быструю `MODEL`.

### 4.1 Каскад

Постройте схему, в которой каждый шаг — **поле** одного класса. Модель заполняет поля по порядку: когда она генерирует шаг 2, она уже видит результат шага 1. **Один вызов, одна схема.**

Опишите класс `TravelPlanCascade` из трёх шагов:

**Шаг 1 — `request: TravelRequest`**: разобрать запрос пользователя
- `destination`: str, `num_days`: int
- `travel_type`: `"adventure"`, `"relaxation"` или `"cultural"`
- `must_visit`: List[str]

**Шаг 2 — `discovery: DestinationInfo`**: знания о месте назначения
- `destination`: str
- `attractions`: List[Attraction] — у каждой `name`, `category`, `typical_duration_hours`, `description`
- `best_area_to_stay`: str, `local_tips`: List[str]

**Шаг 3 — `itinerary: Itinerary`**: план по дням из найденных достопримечательностей
- `destination`: str
- `days`: List[DayPlan] — у каждого `day_number`, `morning` (Activity), `afternoon` (Activity), `evening` (Activity), `total_active_hours`

Вспомогательные модели: `Attraction` (name, category, typical_duration_hours, description), `Activity` (name, duration_hours, category), `DayPlan` (day_number, morning, afternoon, evening, total_active_hours).

Итоговая схема:
```python
class TravelPlanCascade(BaseModel):
    request: TravelRequest
    discovery: DestinationInfo
    itinerary: Itinerary
```

Один вызов `call_neuraldeep(..., response_format=TravelPlanCascade)`.

In [36]:
from typing import List, Literal
from pydantic import Field


class Attraction(StrictBaseModel):
    name: str = Field(
        description="Название достопримечательности"
    )
    category: str = Field(
        description="Категория достопримечательности"
    )
    typical_duration_hours: float = Field(
        description="Типичная продолжительность посещения в часах"
    )
    description: str = Field(
        description="Краткое описание"
    )


class Activity(StrictBaseModel):
    name: str = Field(
        description="Название активности"
    )
    duration_hours: float = Field(
        description="Продолжительность в часах"
    )
    category: str = Field(
        description="Категория активности"
    )


class DayPlan(StrictBaseModel):
    day_number: int

    # ВАЖНО: без Field(description=...)
    # потому что Activity превращается в $ref
    morning: Activity
    afternoon: Activity
    evening: Activity

    total_active_hours: float = Field(
        description="Общее число часов активностей за день"
    )


class TravelRequest(StrictBaseModel):
    destination: str
    num_days: int

    travel_type: Literal[
        "adventure",
        "relaxation",
        "cultural"
    ]

    must_visit: List[str]


class DestinationInfo(StrictBaseModel):
    destination: str

    attractions: List[Attraction]

    best_area_to_stay: str
    local_tips: List[str]


class Itinerary(StrictBaseModel):
    destination: str
    days: List[DayPlan]


class TravelPlanCascade(StrictBaseModel):
    # Тоже без Field(description=...),
    # потому что это ссылки на вложенные модели
    request: TravelRequest
    discovery: DestinationInfo
    itinerary: Itinerary

In [37]:
user_request = "Хочу 3 дня культурной программы в Санкт-Петербурге. Обязательно — Эрмитаж и Петергоф."

# TODO: ОДИН вызов с TravelPlanCascade в response_format

# result = call_neuraldeep(THINKING_MODEL, user_request, response_format=TravelPlanCascade)
# result = call_mistral("ministral-14b-latest", user_request, response_format=TravelPlanCascade)

# pretty_print(result)

result = call_openrouter_pydantic(
    model=THINKING_MODEL,  
    prompt=user_request,
    system_prompt="""
Ты планировщик путешествий.

Заполни схему TravelPlanCascade последовательно:
1. Разбери запрос пользователя.
2. Подбери подходящие достопримечательности.
3. Составь реалистичный план по дням.

Обязательно:
- план должен быть ровно на указанное число дней;
- все места из must_visit должны попасть в итоговый itinerary;
- учитывай тип поездки пользователя;
- используй реальные достопримечательности.
""",
    response_model=TravelPlanCascade
)

pretty_print(result)

TravelPlanCascade:
  ├── request:
    ├── destination: Санкт-Петербург
    ├── num_days: 3
    ├── travel_type: cultural
    ├── must_visit:
      ├── [0]: Эрмитаж
      ├── [1]: Петергоф
  ├── discovery:
    ├── destination: Санкт-Петербург
    ├── attractions:
      ├── [0]:
        ├── name: Эрмитаж
        ├── category: Музей
        ├── typical_duration_hours: 4.0
        ├── description: Один из крупнейших и старейших музеев в мире, расположенный в Зимнем дворце.
      ├── [1]:
        ├── name: Петергоф
        ├── category: Дворцово-парковый ансамбль
        ├── typical_duration_hours: 5.0
        ├── description: Знаменитый своим фонтанным каскадом и красивыми садами дворец.
      ├── [2]:
        ├── name: Исаакиевский собор
        ├── category: Собор
        ├── typical_duration_hours: 2.0
        ├── description: Импозантный собор с великолепным видом на город с его колоннады.
      ├── [3]:
        ├── name: Русский музей
        ├── category: Музей
        ├── typical_du

### 4.2 Маршрутизация

Теперь **расширьте** каскад требованиями, зависящими от типа поездки. У разных типов разные ограничения — маршрутизация добавляет слой требований, который выбирается по типу поездки.

Идея: тот же каскад (запрос → знания → план), но с полем `requirements`, которое через `Union` выбирает нужную схему требований — как `ClaimRouter` в семинаре.

**`AdventureRequirements`** (`kind: Literal["adventure"]`):
- `physical_intensity_per_day`: int (1–5), `outdoor_ratio`: float (0.0–1.0)
- `equipment_needed`: List[str], `best_season`: str

**`RelaxationRequirements`** (`kind: Literal["relaxation"]`):
- `max_activities_per_day`: int (1–2 для спокойного отдыха)
- `downtime_hours_per_day`: float, `spa_or_wellness`: bool

**`CulturalRequirements`** (`kind: Literal["cultural"]`):
- `museums_count`: int, `historical_sites_count`: int
- `local_experiences`: List[str]

**`TravelPlanWithRequirements`** — каскад с маршрутизацией:
```python
class TravelPlanWithRequirements(BaseModel):
    request: TravelRequest
    requirements: Union[AdventureRequirements, RelaxationRequirements, CulturalRequirements]
    discovery: DestinationInfo
    itinerary: Itinerary
```

Модель сначала разбирает запрос, затем выбирает схему требований (маршрутизация), затем подбирает места и планирует с учётом требований — **всё за один вызов**.

Проверьте на трёх запросах (ожидаемые ветки: adventure, relaxation, cultural).

In [38]:
# TODO: опишите схемы требований и TravelPlanWithRequirements
# Переиспользуйте TravelRequest, DestinationInfo, Itinerary из 4.1

# ВАШИ СХЕМЫ

from typing import List, Literal, Union


class AdventureRequirements(StrictBaseModel):
    kind: Literal["adventure"]

    physical_intensity_per_day: int
    outdoor_ratio: float
    equipment_needed: List[str]
    best_season: str


class RelaxationRequirements(StrictBaseModel):
    kind: Literal["relaxation"]

    max_activities_per_day: int
    downtime_hours_per_day: float
    spa_or_wellness: bool


class CulturalRequirements(StrictBaseModel):
    kind: Literal["cultural"]

    museums_count: int
    historical_sites_count: int
    local_experiences: List[str]


class TravelPlanWithRequirements(StrictBaseModel):
    request: TravelRequest

    requirements: Union[
        AdventureRequirements,
        RelaxationRequirements,
        CulturalRequirements
    ]

    discovery: DestinationInfo
    itinerary: Itinerary

In [39]:
test_requests = [
    "Спланируй 4 активных дня в Петербурге и окрестностях: сапы по рекам и каналам, велопрогулки, скалодром и однодневный поход по Карельскому перешейку.",
    "Нужны 5 спокойных дней в Петербурге: спа, неспешные прогулки по паркам и набережным, никакой беготни.",
    "3 дня по музеям и историческим местам Петербурга.",
]

routing_system_prompt = """
Ты планировщик путешествий.

Заполняй схему последовательно:

1. Разбери запрос и определи travel_type:
   - adventure — активный отдых, спорт, походы, outdoor;
   - relaxation — спокойный отдых, спа, прогулки, минимум нагрузки;
   - cultural — музеи, история, культура.

2. Выбери ТОЛЬКО соответствующую схему requirements:
   - adventure -> AdventureRequirements
   - relaxation -> RelaxationRequirements
   - cultural -> CulturalRequirements

Поле kind обязательно должно совпадать с travel_type.

3. Затем подбери достопримечательности и активности,
   учитывая выбранные requirements.

4. Составь itinerary ровно на указанное количество дней.
"""

# Ожидаемые ветки: adventure, relaxation, cultural

for request in test_requests:
    result = call_openrouter_pydantic(
        model=THINKING_MODEL, 
        prompt=request,
        system_prompt=routing_system_prompt,
        response_model=TravelPlanWithRequirements
    )

    print(f"\nЗапрос: {request}")
    print(f"Ветка: {result.requirements.kind}")
    print(f"Тип requirements: {type(result.requirements).__name__}")
    print("-" * 70)


Запрос: Спланируй 4 активных дня в Петербурге и окрестностях: сапы по рекам и каналам, велопрогулки, скалодром и однодневный поход по Карельскому перешейку.
Ветка: adventure
Тип requirements: AdventureRequirements
----------------------------------------------------------------------

Запрос: Нужны 5 спокойных дней в Петербурге: спа, неспешные прогулки по паркам и набережным, никакой беготни.
Ветка: relaxation
Тип requirements: RelaxationRequirements
----------------------------------------------------------------------

Запрос: 3 дня по музеям и историческим местам Петербурга.
Ветка: cultural
Тип requirements: CulturalRequirements
----------------------------------------------------------------------


### 4.3 Цикл

Теперь **расширьте** схему проверкой и итеративным улучшением. Модель предлагает план, проверяет его на ограничения, находит проблемы и улучшает — **всё в одном структурированном ответе** со списком итераций (как `IterativeClaimDecision` в семинаре).

**Ограничения:**
- Все обязательные места включены в план
- Ни в один день активностей не больше 10 часов
- Нет двух выматывающих активностей подряд
- Активности одного дня географически рядом

**`ConstraintCheck`**:
- `all_must_visit_included`: bool, `missing_places`: List[str]
- `any_day_over_10_hours`: bool, `over_limit_days`: List[int]
- `back_to_back_exhausting`: bool, `exhausting_pairs`: List[str]
- `geographically_reasonable`: bool, `geographic_issues`: List[str]
- `all_constraints_passed`: bool

**`PlanIteration`** — один цикл «предложить → проверить → найти исправления»:
- `proposed_plan`: List[DayPlan]
- `constraint_check`: ConstraintCheck
- `issues_found`: List[str], `fixes_to_apply`: List[str]
- `iteration_quality`: int (1–10)

**`TravelPlanWithCycle`** — полная схема:
```python
class TravelPlanWithCycle(BaseModel):
    request: TravelRequest
    requirements: Union[AdventureRequirements, RelaxationRequirements, CulturalRequirements]
    discovery: DestinationInfo
    iterations: List[PlanIteration]  # цикл — несколько раундов улучшения
    final_plan: List[DayPlan]
    all_constraints_met: bool
```

Один вызов: модель заполняет запрос → требования → знания → затем несколько итераций плана, каждый раз проверяя ограничения и улучшая план.

In [40]:
# TODO: опишите ConstraintCheck, PlanIteration и TravelPlanWithCycle
# Переиспользуйте схемы из 4.1 и 4.2

# ВАШИ СХЕМЫ

from typing import List, Union


class ConstraintCheck(StrictBaseModel):
    all_must_visit_included: bool
    missing_places: List[str]

    any_day_over_10_hours: bool
    over_limit_days: List[int]

    back_to_back_exhausting: bool
    exhausting_pairs: List[str]

    geographically_reasonable: bool
    geographic_issues: List[str]

    all_constraints_passed: bool


class PlanIteration(StrictBaseModel):
    proposed_plan: List[DayPlan]

    constraint_check: ConstraintCheck

    issues_found: List[str]
    fixes_to_apply: List[str]

    iteration_quality: int


class TravelPlanWithCycle(StrictBaseModel):
    request: TravelRequest

    requirements: Union[
        AdventureRequirements,
        RelaxationRequirements,
        CulturalRequirements
    ]

    discovery: DestinationInfo

    # Несколько раундов: план -> проверка -> исправления
    iterations: List[PlanIteration]

    final_plan: List[DayPlan]

    all_constraints_met: bool

In [41]:
cycle_request = """
Спланируй 3 дня в Санкт-Петербурге. Тип поездки: культура + гастрономия.
Обязательно: Эрмитаж, Петергоф, Исаакиевский собор, Новая Голландия.

Ограничения:
- Не больше 10 часов активностей в день
- Не ставить две длинные пешие экскурсии подряд
- Активности одного дня — в соседних районах
- Все 4 обязательных места должны быть в плане

Сделай 3 итерации, каждый раз улучшая план.
"""

# TODO: ОДИН вызов с TravelPlanWithCycle

# result = call_neuraldeep(THINKING_MODEL, cycle_request, response_format=TravelPlanWithCycle)
# result = call_mistral("ministral-14b-latest", cycle_request, response_format=TravelPlanWithCycle)
cycle_system_prompt = """
Ты планировщик путешествий.

Нужно выполнить задачу как итеративный цикл в ОДНОМ structured response.

Работай последовательно:

1. request
Разбери запрос пользователя.
Если поездка сочетает культуру и гастрономию, но схема допускает
только adventure / relaxation / cultural, выбери cultural.

2. requirements
Выбери соответствующую ветку требований.
Поле kind должно совпадать с travel_type.

3. discovery
Подбери реальные места Санкт-Петербурга.
Все обязательные места должны быть учтены.

4. iterations
Сделай РОВНО 3 итерации.

В каждой итерации:
- предложи полный план на 3 дня;
- проверь каждое ограничение;
- перечисли реальные найденные проблемы;
- укажи конкретные исправления;
- следующая итерация должна учитывать исправления предыдущей.

Проверяй:
- включены ли все must_visit;
- нет ли дней более 10 часов;
- нет ли двух выматывающих пеших активностей подряд;
- находятся ли активности одного дня географически разумно близко.

Не придумывай проблемы только ради того, чтобы показать изменение.
Если ограничение действительно выполнено, отметь его как выполненное.

5. final_plan
Возьми лучший итоговый вариант после последней итерации.

6. all_constraints_met
true только если в final_plan реально выполнены ВСЕ ограничения.
"""


result = call_openrouter_pydantic(
    model=THINKING_MODEL, 
    prompt=cycle_request,
    system_prompt=cycle_system_prompt,
    response_model=TravelPlanWithCycle
)

pretty_print(result)

TravelPlanWithCycle:
  ├── request:
    ├── destination: Санкт-Петербург
    ├── num_days: 3
    ├── travel_type: cultural
    ├── must_visit:
      ├── [0]: Эрмитаж
      ├── [1]: Петергоф
      ├── [2]: Исаакиевский собор
      ├── [3]: Новая Голландия
  ├── requirements:
    ├── kind: cultural
    ├── museums_count: 2
    ├── historical_sites_count: 2
    ├── local_experiences:
      ├── [0]: Russian Ballet
      ├── [1]: Culinary Masterclass
  ├── discovery:
    ├── destination: Санкт-Петербург
    ├── attractions:
      ├── [0]:
        ├── name: Эрмитаж
        ├── category: Museum
        ├── typical_duration_hours: 4.0
        ├── description: Мировоизвестный музей искусства и культуры.
      ├── [1]:
        ├── name: Петергоф
        ├── category: Historical Site
        ├── typical_duration_hours: 5.0
        ├── description: Императорская резиденция и парк.
      ├── [2]:
        ├── name: Исаакиевский собор
        ├── category: Historical Site
        ├── typical_duration

**TODO:** 2–3 предложения: как менялся план от итерации к итерации? Нашла ли модель реальные проблемы или «проблемы для галочки»? Совпадает ли `all_constraints_met` с тем, что вы видите в `final_plan`?

*Ваш ответ:*

На первой итерации модель обнаружила проблему с двумя длинными активностями подряд и во второй итерации переставила Русский музей на третий день; третья итерация уже ничего не изменила. При этом часть проверки получилась формальной: например, связку "Эрмитаж + Русский музей" она посчитала нарушением, а более заметную географическую проблему второго дня - Петергоф вместе с Исаакиевским собором и Новой Голландией - не отметила. Поэтому all_constraints_met=True я бы не считал полностью корректным: по времени и обязательным местам план проходит, но условие географической близости выполнено не везде.

---
# Задача 5: своя SGR-схема — поездка компании друзей (2.5 балла)

Четверо друзей с **разными интересами** едут вместе на 5 дней. Спроектируйте свою SGR-схему, по которой модель составит план, где всем хорошо, — **за один вызов LLM**.

### Компания
- **Алекс**: хочет приключений — походы, водные виды спорта, всё на свежем воздухе
- **Саша**: любит культуру — музеи, исторические места, местные традиции
- **Женя**: всё про еду — кулинарные мастер-классы, гастротуры, местные рестораны
- **Лера**: нужен отдых — пляжи, спа, красивые виды, без спешки

### Направление: Санкт-Петербург (5 дней)
*(Можно выбрать любой другой город — просто поменяйте переменную `destination` ниже)*

### Требования
- У каждого **минимум 2 активности** по основному интересу
- Каждый день — **минимум 1 общая активность**, которая понравится всем
- Не больше 10 часов активностей в день
- План сбалансирован — ничьи интересы не доминируют

### Что сделать
Спроектируйте **один класс верхнего уровня** `GroupTripPlan`, который через вложенную структуру ведёт модель по всем шагам рассуждения. Помните: SGR — это **один вызов, одна схема**; структура сама управляет рассуждением.

Вызов LLM и печать результата уже написаны ниже — вам нужно только описать схему.

**Подсказка:** какие шаги рассуждения помогут? Например: сначала разобрать потребности каждого, потом подобрать активности в городе, потом найти общие активности, потом построить план по дням, потом проверить, что все довольны. Каждый шаг — вложенное поле класса верхнего уровня.

In [43]:
# TODO: опишите Pydantic-схемы для планировщика групповой поездки.
# Класс верхнего уровня GroupTripPlan должен содержать вложенные классы,
# которые ведут модель по шагам рассуждения.
#
# Пример структуры (можете полностью поменять):
# - Шаг 1: разобрать предпочтения каждого участника
# - Шаг 2: подобрать активности в городе под каждого
# - Шаг 3: найти активности для всей компании
# - Шаг 4: построить план по дням: личные + общие активности
# - Шаг 5: проверить, что у каждого достаточно подходящих активностей
#
# ВАЖНО: класс верхнего уровня должен называться GroupTripPlan
# (он используется в ячейке ниже)

# ВАШИ СХЕМЫ

from typing import List, Literal


FriendName = Literal["Алекс", "Саша", "Женя", "Лера"]

InterestType = Literal[
    "adventure",
    "culture",
    "food",
    "relaxation"
]


# =========================
# ШАГ 1. Предпочтения
# =========================

class FriendPreference(StrictBaseModel):
    name: FriendName
    primary_interest: InterestType
    desired_activities: List[str]
    important_constraints: List[str]


class PreferenceAnalysis(StrictBaseModel):
    destination: str
    num_days: int
    participants: List[FriendPreference]


# =========================
# ШАГ 2. Подбор активностей
# =========================

class CandidateActivity(StrictBaseModel):
    name: str
    category: InterestType
    duration_hours: float
    location_area: str

    suitable_for: List[FriendName]
    why_suitable: str


class ActivityDiscovery(StrictBaseModel):
    activities: List[CandidateActivity]


# =========================
# ШАГ 3. Общие активности
# =========================

class SharedActivity(StrictBaseModel):
    name: str
    duration_hours: float
    location_area: str
    why_everyone_will_enjoy_it: str


class SharedActivitySelection(StrictBaseModel):
    shared_activities: List[SharedActivity]


# =========================
# ШАГ 4. План по дням
# =========================

class PlannedActivity(StrictBaseModel):
    name: str
    category: InterestType
    duration_hours: float

    participants: List[FriendName]

    is_shared: bool
    location_area: str


class GroupDayPlan(StrictBaseModel):
    day_number: int

    activities: List[PlannedActivity]

    total_hours: float

    # Так явно заставляем модель показать,
    # какая общая активность есть в этот день
    shared_activity_name: str


class GroupItinerary(StrictBaseModel):
    days: List[GroupDayPlan]


# =========================
# ШАГ 5. Проверка
# =========================

class ParticipantCoverage(StrictBaseModel):
    name: FriendName

    matching_activities: List[str]
    matching_activities_count: int

    has_at_least_two: bool


class DayValidation(StrictBaseModel):
    day_number: int

    total_hours: float
    within_10_hours: bool

    has_shared_activity: bool


class BalanceValidation(StrictBaseModel):
    adventure_count: int
    culture_count: int
    food_count: int
    relaxation_count: int

    balanced: bool
    explanation: str


class GroupTripValidation(StrictBaseModel):
    participant_coverage: List[ParticipantCoverage]

    day_checks: List[DayValidation]

    balance: BalanceValidation

    issues_found: List[str]

    all_requirements_met: bool


# =========================
# ВСЯ SGR-СХЕМА
# =========================

class GroupTripPlan(StrictBaseModel):

    # Шаг 1
    preferences: PreferenceAnalysis

    # Шаг 2
    discovery: ActivityDiscovery

    # Шаг 3
    shared_activity_selection: SharedActivitySelection

    # Шаг 4
    itinerary: GroupItinerary

    # Шаг 5
    validation: GroupTripValidation

In [44]:
destination = "Воркута"  # <-- можно поменять на любой город

user_request = f"""
Спланируй 5-дневную поездку в {destination} для компании из 4 друзей:
- Алекс: хочет приключений (походы, водные виды спорта, свежий воздух)
- Саша: любит культуру (музеи, исторические места, местные традиции)
- Женя: всё про еду (кулинарные мастер-классы, гастротуры, местные рестораны)
- Лера: нужен отдых (пляжи, спа, красивые виды, без спешки)

Требования:
- У каждого минимум 2 активности по его интересам
- Каждый день — минимум 1 общая активность, которая понравится всем
- Не больше 10 часов активностей в день
- План сбалансирован — ничьи интересы не доминируют
"""

group_system_prompt = """
Ты планировщик групповых путешествий.

Заполняй схему строго последовательно.

ШАГ 1 — preferences:
проанализируй интересы каждого участника.

ШАГ 2 — discovery:
подбери реальные активности в указанном городе.
У каждого участника должно быть достаточно вариантов
по его основному интересу.

ШАГ 3 — shared_activity_selection:
найди активности, которые подходят сразу всем четырём друзьям.
Нужно достаточно вариантов, чтобы минимум одна такая
активность попала в каждый день поездки.

ШАГ 4 — itinerary:
составь план РОВНО на 5 дней.

Обязательно:
- у каждого минимум 2 активности по его основному интересу;
- каждый день содержит хотя бы 1 общую активность;
- не больше 10 часов активностей в день;
- интересы участников распределены сбалансированно.

ШАГ 5 — validation:
проверь итоговый itinerary по каждому ограничению.

Не ставь all_requirements_met=true,
если хотя бы одно ограничение фактически нарушено.
"""


#result = call_neuraldeep(THINKING_MODEL, user_request, response_format=GroupTripPlan)
# result = call_mistral("ministral-14b-latest", user_request, response_format=GroupTripPlan)

result = call_openrouter_pydantic(
    model=THINKING_MODEL,   
    prompt=user_request,
    system_prompt=group_system_prompt,
    response_model=GroupTripPlan
)

pretty_print(result)

GroupTripPlan:
  ├── preferences:
    ├── destination: Воркута
    ├── num_days: 5
    ├── participants:
      ├── [0]:
        ├── name: Алекс
        ├── primary_interest: adventure
        ├── desired_activities:
          ├── [0]: походы в горы
          ├── [1]: водные виды спорта
        ├── important_constraints:
          ├── [0]: активный отдых
      ├── [1]:
        ├── name: Саша
        ├── primary_interest: culture
        ├── desired_activities:
          ├── [0]: посещение музеев
          ├── [1]: исторические места
        ├── important_constraints:
          ├── [0]: познавательные экскурсии
      ├── [2]:
        ├── name: Женя
        ├── primary_interest: food
        ├── desired_activities:
          ├── [0]: гастрономические туры
          ├── [1]: кулинарные мастер-классы
        ├── important_constraints:
          ├── [0]: местная кухня
      ├── [3]:
        ├── name: Лера
        ├── primary_interest: relaxation
        ├── desired_activities:
          ├── 

### Проверка

**TODO:** проверьте результат выше и ответьте:
1. Есть ли у каждого (Алекс, Саша, Женя, Лера) минимум 2 подходящие активности?
2. Есть ли в каждом дне хотя бы 1 общая активность?
3. Есть ли день больше 10 часов активностей?
4. Сбалансирован ли план по интересам?
5. Если в схеме есть шаг проверки — совпадают ли его выводы с тем, что видите вы? Какой шаг схемы сильнее всего повлиял на качество плана?

*Ваш ответ:*


1. По автоматической проверке у каждого есть минимум по 2 подходящие активности, но вручную видно нюанс: у Алекса второй активностью засчитана «Экскурсия в природный заповедник», хотя в плане она помечена как culture, а не adventure. Поэтому это условие выполнено не совсем строго.  
2. Да.    
3. Дней длиннее 10 часов нет: продолжительность 9.0, 8.5, 9.0, 6.5 и 9.0 часов.
4. В целом план достаточно сбалансирован: количество активностей по категориям близкое - adventure=2, culture=3, food=3, relaxation=3.
5. Шаг валтдации в основном правильно проверил время и наличие общих активностей, но слишком оптимистично выставил all_requirements_met=True. Сильнее всего на качество повлиял именно отдельный шаг проверки, потому что он заставил модель явно посчитать покрытие интересов каждого участника и ограничения по каждому дню, хотя его выводы всё равно полезно перепроверять вручную.